Building an interactive AI agent with LangChain that interprets natural language queries and dynamically selects and executes tools to perform specific tasks.

### Installing Required Libraries


In [ ]:
%pip install langchain===0.3.25 | tail -n 1
%pip install langchain-openai===0.3.19 | tail -n 1

### Importing Required Libraries


In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

Initializing GPT-4o-mini through LangChain’s OpenAI integration to process user queries and determine which tools to call.

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

# API Disclaimer


In [ ]:

from langchain_openai import ChatOpenAI
from langchain_ibm import ChatWatsonx
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key = "openai api key",
)
watsonx_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="project id associated with the API key",
    api_key="watsonx.ai api key",
)

## Creating Custom Tools with LangChain

Creating custom tools with the @tool decorator, descriptive function names, type annotations, and clear docstrings to guide tool selection and input validation. Each function implements the operation, handles errors, and returns results for the model to use.



In [3]:
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b

### Add tools to the LLM

cConnecting and binding the function to the chat model.


In [4]:
tools = [add]

llm_with_tools = llm.bind_tools(tools)

Creating some more basic arithmetic tools.


In [5]:
@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b

### Testing the functions


In [6]:
tool_map = {
    "add": add, 
    "subtract": subtract,
    "multiply": multiply
}

input_ = {
    "a": 1,
    "b": 2
}

tool_map["add"].invoke(input_)

3

Testing each custom tool with .invoke(inputs) to verify its behavior and output across different inputs.

### Adding new tools to LLM


In [7]:
tools = [add, subtract, multiply]

llm_with_tools = llm.bind_tools(tools)

## Interacting with the Model


In [13]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

Running the model with the context (chat history) that contains the user query.


In [14]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print(type(response_1))
#print(response_1)

<class 'langchain_core.messages.ai.AIMessage'>

Parsing the model’s response to identify requested tools and extract their input arguments.

In [15]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name:\n{tool_1_name}')
print(f'tool args:\n{tool_1_args}')
print(f'tool call ID:\n{tool_call_1_id}')

tool name:
add
tool args:
{'a': 3, 'b': 2}
tool call ID:
call_r7jbFaaJkINeIN5dfAVglfcT

Executing the requested tool with the arguments provided in the model’s tool call.

In [16]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

content='5' tool_call_id='call_r7jbFaaJkINeIN5dfAVglfcT'

In [17]:
chat_history.append(tool_message)

### Generating a final answer from chat history



In [18]:
answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
The result of \(3 + 2\) is \(5\).

## Building an Agent


In [ ]:
class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.contet 
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

In [20]:
my_agent = ToolCallingAgent(llm)

print(my_agent.run("one plus 2"))

print(my_agent.run("one - 2"))

print(my_agent.run("three times two"))

One plus two equals three.
The result of \(1 - 2\) is \(-1\).
Three times two is 6.

### Validating tool creation and LLM tool calling

Creating a calculate_tip tool that accepts total_bill and tip_percent, testing it directly with 120 and 15 to verify a tip of $18, and adding it to tool_map.

Binding the tool to the model and testing the query, “How much should I tip on $60 at 20%?” Parsing the model’s tool call, executing the calculation, and passing the complete message history back to the model to generate a final response with the expected tip of $12.



In [ ]:

@tool
def calculate_tip(total_bill: int, tip_percent: int ) -> int:
    """ Calculate tip"""
    return total_bill * tip_percent * 0.01
inputs = {
    "total_bill" : 120,
    "tip_percent" : 15  
} 
calculate_tip.invoke(inputs)
tool_map = {
    "calculate_tip" : calculate_tip
    
}


In [ ]:

query = "How much should I tip on $60 at 20%?"

llm_with_tool = llm.bind_tools([calculate_tip])
chat_history = [HumanMessage(content=query)]

response = llm_with_tool.invoke(chat_history)

tool_calls = response.tool_calls
tool_name = tool_calls[0]["name"]
tool_args = tool_calls[0]["args"]
tool_call_id = tool_calls[0]["id"]

tool_response = tool_map[tool_name].invoke(tool_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_id)

chat_history.extend([response, tool_message])
result = llm_with_tool.invoke(chat_history)
print(result.content)

You should tip $12 on a $60 bill at 20%.

### Automating the tip calculation workflow

Building an agent to automate tool selection, execution, and response generation for the query, “How much should I tip on $60 at 20%?” Verifying that it calls calculate_tip with the correct arguments and returns the expected tip of $12.


In [ ]:

query = "How much should I tip on $60 at 20%?"

class TipAgent:
    def __init__(self, llm):
        self.llm_with_tool = llm.bind_tools([calculate_tip])
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        chat_history = [HumanMessage(content=query)]
        response = llm_with_tool.invoke(chat_history)

        tool_calls = response.tool_calls
        tool_name = tool_calls[0]["name"]
        tool_args = tool_calls[0]["args"]
        tool_call_id = tool_calls[0]["id"]

        tool_response = tool_map[tool_name].invoke(tool_args)
        tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_id)
        
        chat_history.extend([response, tool_message])
        
        return llm_with_tool.invoke(chat_history).content

    

agent = TipAgent(llm)
#agent.run(query)
agent.run("How much should I tip on $60 at 20%?")
        

        

'You should tip $12 on a $60 bill at 20%.'

<!-- ## Changelog

| Date | Version | Changed by | Change Description |
|------|--------|--------|---------|
| 2024-06-06 | 0.1 |  P. Kravitz | ID review and edit. No code edits.Updated the copyright statement. Change log added. Instructional edits only for IBM style. Second person, accessibility, and other minor grammar edits.| -->
